# MORA12, SALPRES, VEA y SINMORA

In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joblib

In [2]:
def get_data_path(relative_path: str = "") -> Path:
    """Retorna la ruta absoluta a la carpeta de datos compartida."""
    # Busca la raíz del repositorio basándose en este archivo (src/config.py)
    root = Path.cwd().parent.parent
    data_dir = root / relative_path
    
    if not data_dir.exists():
        os.makedirs(data_dir)
        
    return data_dir

In [3]:
# Cargar informes de cartera de los últimos 12 meses
data_dir = get_data_path('data/data-09-2025/informes-cartera')
archivos = os.listdir(data_dir)
dataframes = {}
for archivo in archivos:
    if archivo.endswith('.xlsx'):
        nombre_df = archivo.split('.')[0]
        nombre_df = nombre_df.split('_', maxsplit=1)[1]
        ruta_archivo = data_dir / archivo
        dataframes[nombre_df] = pd.read_excel(ruta_archivo, header=3)
        print(f'Archivo {archivo} cargado como DataFrame {nombre_df}')

Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_ABRIL_2025.xlsx cargado como DataFrame ABRIL_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_AGOSTO_2025.xlsx cargado como DataFrame AGOSTO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_DICIEMBRE_2024.xlsx cargado como DataFrame DICIEMBRE_2024
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_ENERO_2025.xlsx cargado como DataFrame ENERO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_FEBRERO_2025.xlsx cargado como DataFrame FEBRERO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_JULIO_2025.xlsx cargado como DataFrame JULIO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_JUNIO_2025.xlsx cargado como DataFrame JUNIO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_MARZO_2025.xlsx cargado como DataFrame MARZO_2025
Archivo INFORME INDIVIDUAL DE CARTERA DE CREDITO (MODIFICADO)_MAYO_2025.xlsx cargado como 

In [4]:
# Preservar en cada dataframe solo las columnas "NIT", "NroCredito", "CodigoContable" y "Morosidad"
for nombre_df in dataframes.keys():
    dataframes[nombre_df] = dataframes[nombre_df][['NIT', 'NroCredito', 'CodigoContable', 'Morosidad','ValorPrestamo', 'SaldoCapital', 'SaldoIntereses', 'OtrosSaldos', 'clasegarantia']]
    dataframes[nombre_df] = dataframes[nombre_df].dropna(subset=['NroCredito'])
    dataframes[nombre_df] = dataframes[nombre_df].drop_duplicates(subset=['NroCredito'])
    dataframes[nombre_df] = dataframes[nombre_df].set_index('NroCredito')
    dataframes[nombre_df]['SALPRES'] = dataframes[nombre_df]['SaldoCapital']/dataframes[nombre_df]['ValorPrestamo']
    dataframes[nombre_df]['VEA'] = dataframes[nombre_df]['SaldoCapital']+dataframes[nombre_df]['SaldoIntereses']+dataframes[nombre_df]['OtrosSaldos']

# Hacer una lista con las keys de los dataframes ordenadas cronológicamente de forma automática
meses_es = {
	'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
	'JULIO': 7, 'AGOSTO': 8, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
}

def mes_ano_to_date(mes_ano):
	mes, ano = mes_ano.split('_')
	return pd.Timestamp(year=int(ano), month=meses_es[mes], day=1)

meses_ordenados = sorted(dataframes.keys(), key=mes_ano_to_date)
meses_ordenados

['OCTUBRE_2024',
 'NOVIEMBRE_2024',
 'DICIEMBRE_2024',
 'ENERO_2025',
 'FEBRERO_2025',
 'MARZO_2025',
 'ABRIL_2025',
 'MAYO_2025',
 'JUNIO_2025',
 'JULIO_2025',
 'AGOSTO_2025',
 'SEPTIEMBRE_2025']

In [5]:
# Fusionar todos los dataframes en uno solo, usando como clave primaria el índice, usando la función merge el método outer, y nombrando cada columna nueva con el nombre del dataframe original
df_mora = dataframes[meses_ordenados[0]][['NIT', 'Morosidad', 'CodigoContable', 'SALPRES', 'VEA']].copy()
df_mora = df_mora.rename(columns={'Morosidad': f'Morosidad_{meses_ordenados[0]}', 
                                  'CodigoContable': f'CodigoContable_{meses_ordenados[0]}', 
                                  'NIT': f'NIT_{meses_ordenados[0]}',
                                  'SALPRES': f'SALPRES_{meses_ordenados[0]}',
                                  'VEA': f'VEA_{meses_ordenados[0]}',
                                  'clasegarantia': f'clasegarantia_{meses_ordenados[0]}'}
                                  )

for mes in meses_ordenados[1:]:
    df_temp = dataframes[mes][['NIT', 'Morosidad', 'CodigoContable', 'SALPRES', 'VEA', 'clasegarantia']].copy()
    df_temp = df_temp.rename(columns={'Morosidad': f'Morosidad_{mes}', 
                                      'CodigoContable': f'CodigoContable_{mes}', 
                                      'NIT': f'NIT_{mes}',
                                      'SALPRES': f'SALPRES_{mes}',
                                      'VEA': f'VEA_{mes}',
                                      'clasegarantia': f'clasegarantia_{mes}'}
                                      )
    
    df_mora = df_mora.merge(df_temp, left_index=True, right_index=True, how='outer') 
df_mora

,NIT_OCTUBRE_2024,Morosidad_OCTUBRE_2024,CodigoContable_OCTUBRE_2024,SALPRES_OCTUBRE_2024,VEA_OCTUBRE_2024,NIT_NOVIEMBRE_2024,Morosidad_NOVIEMBRE_2024,CodigoContable_NOVIEMBRE_2024,SALPRES_NOVIEMBRE_2024,VEA_NOVIEMBRE_2024,...,CodigoContable_AGOSTO_2025,SALPRES_AGOSTO_2025,VEA_AGOSTO_2025,clasegarantia_AGOSTO_2025,NIT_SEPTIEMBRE_2025,Morosidad_SEPTIEMBRE_2025,CodigoContable_SEPTIEMBRE_2025,SALPRES_SEPTIEMBRE_2025,VEA_SEPTIEMBRE_2025,clasegarantia_SEPTIEMBRE_2025
NroCredito,,,,,,,,,,,,,,,,,,,,,
7,1040039171,0.0,146905.0,0.970262,77620954.0,1040039171,0.0,146905.0,0.955391,76431287.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,1036670861,0.0,146930.0,0.972392,6517916.0,1036670861,0.0,146930.0,0.954423,6397474.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
007-002-0001463-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,144205.0,0.936706,966346.0,1.0,1035912774,44.0,144210.0,0.936706,980626.0,1.0
007-002-0001464-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,144205.0,0.400000,4002147.0,1.0,15437544,0.0,144205.0,0.400000,4066557.0,1.0
007-002-0001467-8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,144205.0,0.887833,2701704.0,1.0,43907525,0.0,144205.0,0.880690,2684521.0,1.0


In [6]:
# Se eliminan todos los registros que no tienen NIT en el último mes, para que solo queden los créditos vigentes al último mes de los datos
last_month = meses_ordenados[-1]
df_mora = df_mora.dropna(subset=[f'NIT_{last_month}'])
df_mora.info()

<class 'pandas.DataFrame'>
Index: 21020 entries, 002-001-0000024-8 to 007-002-0001472-1
Data columns (total 71 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   NIT_OCTUBRE_2024                13293 non-null  object 
 1   Morosidad_OCTUBRE_2024          13293 non-null  float64
 2   CodigoContable_OCTUBRE_2024     13293 non-null  float64
 3   SALPRES_OCTUBRE_2024            13293 non-null  float64
 4   VEA_OCTUBRE_2024                13293 non-null  float64
 5   NIT_NOVIEMBRE_2024              13893 non-null  object 
 6   Morosidad_NOVIEMBRE_2024        13893 non-null  float64
 7   CodigoContable_NOVIEMBRE_2024   13893 non-null  float64
 8   SALPRES_NOVIEMBRE_2024          13893 non-null  float64
 9   VEA_NOVIEMBRE_2024              13893 non-null  float64
 10  clasegarantia_NOVIEMBRE_2024    13893 non-null  float64
 11  NIT_DICIEMBRE_2024              14378 non-null  object 
 12  Morosidad_DICIEMBRE_

In [7]:
# Crear una columna nueva que sea el valor máximo de morosidad en los últimos 12 meses tomando únicamente las columnas que contienen la palabra "Morosidad"
df_mora[f'MORA12_{meses_ordenados[-1]}'] = df_mora.filter(like='Morosidad').max(axis=1, skipna=True)
df_mora # Muestra las primeras filas del DataFrame final

,NIT_OCTUBRE_2024,Morosidad_OCTUBRE_2024,CodigoContable_OCTUBRE_2024,SALPRES_OCTUBRE_2024,VEA_OCTUBRE_2024,NIT_NOVIEMBRE_2024,Morosidad_NOVIEMBRE_2024,CodigoContable_NOVIEMBRE_2024,SALPRES_NOVIEMBRE_2024,VEA_NOVIEMBRE_2024,...,SALPRES_AGOSTO_2025,VEA_AGOSTO_2025,clasegarantia_AGOSTO_2025,NIT_SEPTIEMBRE_2025,Morosidad_SEPTIEMBRE_2025,CodigoContable_SEPTIEMBRE_2025,SALPRES_SEPTIEMBRE_2025,VEA_SEPTIEMBRE_2025,clasegarantia_SEPTIEMBRE_2025,MORA12_SEPTIEMBRE_2025
NroCredito,,,,,,,,,,,,,,,,,,,,,
002-001-0000024-8,901-285-696-7,0.0,146205.0,0.447354,2.244973e+07,901-285-696-7,0.0,146205.0,0.430239,2.155141e+07,...,0.259650,1.300153e+07,1.0,901-285-696-7,0.0,146515.0,0.239269,1.199857e+07,1.0,0.0
002-001-0000026-6,900-698-421-8,8.0,146205.0,0.659054,6.697431e+07,900-698-421-8,8.0,146205.0,0.643515,6.541299e+07,...,0.528942,5.328384e+07,1.0,900-698-421-8,16.0,146515.0,0.528942,5.395877e+07,1.0,38.0
002-001-0000027-5,811-021-864-9,0.0,146205.0,0.787416,1.853138e+09,811-021-864-9,0.0,146205.0,0.777588,1.830738e+09,...,0.685802,1.613352e+09,1.0,811-021-864-9,6.0,146515.0,0.685802,1.632600e+09,1.0,6.0
002-001-0000028-4,900-409-773-7,5.0,146205.0,0.687549,1.399760e+08,900-409-773-7,5.0,146205.0,0.672469,1.369056e+08,...,0.543393,1.113402e+08,1.0,900-409-773-7,5.0,146515.0,0.508649,1.027004e+08,1.0,35.0
002-001-0000029-3,901-406-831-6,30.0,146205.0,0.895916,6.521809e+08,901-406-831-6,1.0,146205.0,0.885822,6.441110e+08,...,0.848534,6.309995e+08,1.0,901-406-831-6,91.0,146525.0,0.848534,6.309995e+08,1.0,91.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
007-002-0001463-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.936706,9.663460e+05,1.0,1035912774,44.0,144210.0,0.936706,9.806260e+05,1.0,44.0
007-002-0001464-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.400000,4.002147e+06,1.0,15437544,0.0,144205.0,0.400000,4.066557e+06,1.0,0.0
007-002-0001467-8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.887833,2.701704e+06,1.0,43907525,0.0,144205.0,0.880690,2.684521e+06,1.0,0.0


In [8]:
# Conservar solo las columnas que contienen el último mes
columnas_a_conservar = [f'Morosidad_{meses_ordenados[-1]}', 
                        f'CodigoContable_{meses_ordenados[-1]}', 
                        f'NIT_{meses_ordenados[-1]}', 
                        f'SALPRES_{meses_ordenados[-1]}', 
                        f'VEA_{meses_ordenados[-1]}', 
                        f'MORA12_{meses_ordenados[-1]}', 
                        f'clasegarantia_{meses_ordenados[-1]}']

df_mora = df_mora[columnas_a_conservar]
df_mora

,Morosidad_SEPTIEMBRE_2025,CodigoContable_SEPTIEMBRE_2025,NIT_SEPTIEMBRE_2025,SALPRES_SEPTIEMBRE_2025,VEA_SEPTIEMBRE_2025,MORA12_SEPTIEMBRE_2025,clasegarantia_SEPTIEMBRE_2025
NroCredito,,,,,,,
002-001-0000024-8,0.0,146515.0,901-285-696-7,0.239269,1.199857e+07,0.0,1.0
002-001-0000026-6,16.0,146515.0,900-698-421-8,0.528942,5.395877e+07,38.0,1.0
002-001-0000027-5,6.0,146515.0,811-021-864-9,0.685802,1.632600e+09,6.0,1.0
002-001-0000028-4,5.0,146515.0,900-409-773-7,0.508649,1.027004e+08,35.0,1.0
002-001-0000029-3,91.0,146525.0,901-406-831-6,0.848534,6.309995e+08,91.0,1.0
...,...,...,...,...,...,...,...
007-002-0001463-2,44.0,144210.0,1035912774,0.936706,9.806260e+05,44.0,1.0
007-002-0001464-1,0.0,144205.0,15437544,0.400000,4.066557e+06,0.0,1.0
007-002-0001467-8,0.0,144205.0,43907525,0.880690,2.684521e+06,0.0,1.0


In [9]:
# Quitar el mes y el año de los nombres de las columnas
new_column_names = {f'Morosidad_{meses_ordenados[-1]}': 'Morosidad',
                        f'CodigoContable_{meses_ordenados[-1]}': 'CodigoContable',
                        f'NIT_{meses_ordenados[-1]}': 'NIT',
                        f'SALPRES_{meses_ordenados[-1]}': 'SALPRES',
                        f'VEA_{meses_ordenados[-1]}': 'VEA',
                        f'MORA12_{meses_ordenados[-1]}': 'MORA12',
                        f'clasegarantia_{meses_ordenados[-1]}': 'clasegarantia'
                        }

df_mora = df_mora.rename(columns=new_column_names)
df_mora = df_mora.reset_index()
df_mora['SINMORA'] = np.where(df_mora['MORA12'] == 0, 1, 0)
df_mora


,NroCredito,Morosidad,CodigoContable,NIT,SALPRES,VEA,MORA12,clasegarantia,SINMORA
0,002-001-0000024-8,0.0,146515.0,901-285-696-7,0.239269,1.199857e+07,0.0,1.0,1
1,002-001-0000026-6,16.0,146515.0,900-698-421-8,0.528942,5.395877e+07,38.0,1.0,0
2,002-001-0000027-5,6.0,146515.0,811-021-864-9,0.685802,1.632600e+09,6.0,1.0,0
3,002-001-0000028-4,5.0,146515.0,900-409-773-7,0.508649,1.027004e+08,35.0,1.0,0
4,002-001-0000029-3,91.0,146525.0,901-406-831-6,0.848534,6.309995e+08,91.0,1.0,0
...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,44.0,144210.0,1035912774,0.936706,9.806260e+05,44.0,1.0,0
21016,007-002-0001464-1,0.0,144205.0,15437544,0.400000,4.066557e+06,0.0,1.0,1
21017,007-002-0001467-8,0.0,144205.0,43907525,0.880690,2.684521e+06,0.0,1.0,1
21018,007-002-0001468-7,14.0,144205.0,1036964056,0.940565,4.851986e+06,14.0,1.0,0


In [10]:
# Crear una nueva columna llamada PDI que se calcula así:
# Si clasegarantia es igual a 1 o a 4, entonces:
#  Si Morosidad >= 420, PDI = 1.0,
#  Si Morosidad < 420 y Morosidad >= 210, PDI = 0.7,
#  Si Morosidad < 210, PDI = 0.6,
#  Si Morosidad < 90, PDI = 0.45,


# Si clasegarantia es igual a 2, entonces:
#  Si Morosidad >= 720, PDI = 1.0,
#  Si Morosidad < 720 y Morosidad >= 360, PDI = 0.7,
#  Si Morosidad < 360, PDI = 0.4,
# Si clasegarantia es igual a 3, entonces:
#  Si Morosidad >= 540, PDI = 1.0,
#  Si Morosidad < 540 y Morosidad >= 270, PDI = 0.7,
#  Si Morosidad < 270, PDI = 0.5

def calcular_pdi(row):
    clase = row['clasegarantia']
    morosidad = row['Morosidad']
    
    if clase in [1, 4]:
        if morosidad >= 420:
            return 1.0
        elif morosidad >= 210:
            return 0.7
        elif morosidad >= 90:
            return 0.6
        else:
            return 0.45
    elif clase == 2:
        if morosidad >= 720:
            return 1.0
        elif morosidad >= 360:
            return 0.7
        else:
            return 0.4
    elif clase == 3:
        if morosidad >= 540:
            return 1.0
        elif morosidad >= 270:
            return 0.7
        else:
            return 0.5
    else:
        return np.nan

df_mora['PDI'] = df_mora.apply(calcular_pdi, axis=1)
df_mora

,NroCredito,Morosidad,CodigoContable,NIT,SALPRES,VEA,MORA12,clasegarantia,SINMORA,PDI
0,002-001-0000024-8,0.0,146515.0,901-285-696-7,0.239269,1.199857e+07,0.0,1.0,1,0.45
1,002-001-0000026-6,16.0,146515.0,900-698-421-8,0.528942,5.395877e+07,38.0,1.0,0,0.45
2,002-001-0000027-5,6.0,146515.0,811-021-864-9,0.685802,1.632600e+09,6.0,1.0,0,0.45
3,002-001-0000028-4,5.0,146515.0,900-409-773-7,0.508649,1.027004e+08,35.0,1.0,0,0.45
4,002-001-0000029-3,91.0,146525.0,901-406-831-6,0.848534,6.309995e+08,91.0,1.0,0,0.60
...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,44.0,144210.0,1035912774,0.936706,9.806260e+05,44.0,1.0,0,0.45
21016,007-002-0001464-1,0.0,144205.0,15437544,0.400000,4.066557e+06,0.0,1.0,1,0.45
21017,007-002-0001467-8,0.0,144205.0,43907525,0.880690,2.684521e+06,0.0,1.0,1,0.45
21018,007-002-0001468-7,14.0,144205.0,1036964056,0.940565,4.851986e+06,14.0,1.0,0,0.45


# Antiguedad (ANTI) y Activo

In [16]:
# Cargar el archivo 'Archivos originales/INFORME ASOCIADOS_JUNIO_2022.xlsx' en un dataframe llamado 'vinculacion_{mes_año}' donde {mes_año} está extraído del nombre del archivo
data_dir = get_data_path('data/data-09-2025')
archivo = data_dir / 'INFORME ASOCIADOS_SEPTIEMBRE_2025.xlsx'
mes_anno = archivo.name.split('.')[0]
mes_anno = mes_anno.split('_', maxsplit=1)[1]
mes_anno

'SEPTIEMBRE_2025'

In [17]:
nombre_df = f'vinculacion_{mes_anno}'
vinculaciones = {}
vinculaciones[nombre_df] = pd.read_excel(
    archivo,
    header=3
)

vinculaciones[nombre_df]['Fecha de ingreso'] = pd.to_datetime(vinculaciones[nombre_df]['Fecha de ingreso'], errors='coerce', format='%d/%m/%Y')
vinculaciones[nombre_df]['Activo'] = vinculaciones[nombre_df]['Activo'].fillna(1).astype(int)
vinculaciones[nombre_df].info()

<class 'pandas.DataFrame'>
RangeIndex: 50239 entries, 0 to 50238
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Número de identificación  50239 non-null  object        
 1   Nombres                   50239 non-null  str           
 2   Primer apellido           49937 non-null  str           
 3   Activo                    50239 non-null  int64         
 4   Fecha de ingreso          50239 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1), object(1), str(2)
memory usage: 1.9+ MB


In [18]:
# Crear una columna nueva llamada 'Activo desde' que contenga el número de días que lleva activo el cliente desde la fecha de ingreso hasta el último día del mes que orresponde a la variable mes_anno. Si 'Activo' es 0 o Nan, entonces esta variable también será 0
ultimo_dia_mes = mes_ano_to_date(mes_anno) + pd.offsets.MonthEnd(0)
vinculaciones[nombre_df]['ANTI'] = vinculaciones[nombre_df]['Activo'] * (ultimo_dia_mes - vinculaciones[nombre_df]['Fecha de ingreso']).dt.days
vinculaciones[nombre_df]

,Número de identificación,Nombres,Primer apellido,Activo,Fecha de ingreso,ANTI
0,96055,Felix Antonio,Castaño,1,2002-05-18,8536
1,152419,Jacques,Fournier,1,2012-02-24,4967
2,179810,Edwin,Arenas,1,2015-12-15,3577
3,204414,Lina Zulema,Pedrazas,1,2019-01-28,2437
4,219552,German Rodrigo,Tenelanda,1,2002-05-25,8529
...,...,...,...,...,...,...
50234,901-792-877-8,Inversiones Saint Polito Sas,NaN,1,2024-02-21,587
50235,1-037-614-537-4,Villegas Tobon Leidy Johana,NaN,1,2022-08-03,1154
50236,825276723041978,Katerine Del Valle,Hernandez,1,2021-06-30,1553
50237,924835703011983,Gabriel Eduardo,Joaquim,1,2020-02-01,2068


In [19]:
# Fusionar el dataframe df_mora con el dataframe vinculaciones[nombre_df] usando como clave primaria la columna 'NIT' de df_mora y la columna 'Número de identificación' de vinculaciones[nombre_df], usando el método left
df_features = pd.merge(
    df_mora[['NroCredito','NIT', 'CodigoContable', 'SALPRES', 'VEA','MORA12','SINMORA', 'Morosidad', 'clasegarantia', 'PDI']], 
    vinculaciones[nombre_df][['Número de identificación', 'Activo', 'ANTI']], 
    left_on='NIT', 
    right_on='Número de identificación', 
    how='left'
)

df_features = df_features.drop(columns=['Número de identificación'])
df_features

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI
0,002-001-0000024-8,901-285-696-7,146515.0,0.239269,1.199857e+07,0.0,1,0.0,1.0,0.45,1.0,1448.0
1,002-001-0000026-6,900-698-421-8,146515.0,0.528942,5.395877e+07,38.0,0,16.0,1.0,0.45,1.0,1181.0
2,002-001-0000027-5,811-021-864-9,146515.0,0.685802,1.632600e+09,6.0,0,6.0,1.0,0.45,1.0,1791.0
3,002-001-0000028-4,900-409-773-7,146515.0,0.508649,1.027004e+08,35.0,0,5.0,1.0,0.45,1.0,1141.0
4,002-001-0000029-3,901-406-831-6,146525.0,0.848534,6.309995e+08,91.0,0,91.0,1.0,0.60,1.0,1131.0
...,...,...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,1035912774,144210.0,0.936706,9.806260e+05,44.0,0,44.0,1.0,0.45,1.0,1457.0
21016,007-002-0001464-1,15437544,144205.0,0.400000,4.066557e+06,0.0,1,0.0,1.0,0.45,1.0,6405.0
21017,007-002-0001467-8,43907525,144205.0,0.880690,2.684521e+06,0.0,1,0.0,1.0,0.45,1.0,463.0
21018,007-002-0001468-7,1036964056,144205.0,0.940565,4.851986e+06,14.0,0,14.0,1.0,0.45,1.0,564.0


# Aportes (AP)

In [20]:
archivo = data_dir / 'INFORME INDIVIDUAL DE APORTES_SEPTIEMBRE_2025.xlsx'
nombre_df = f'aporte_{mes_anno}'
aportes = {}
aportes[nombre_df] = pd.read_excel(
    archivo,
    header=3
)

aportes[nombre_df]['Aporte/Contribución Ordinario'] = aportes[nombre_df]['Aporte/Contribución Ordinario'].fillna(0)
aportes[nombre_df] = aportes[nombre_df].rename(columns={'Aporte/Contribución Ordinario': 'AP'})  
aportes[nombre_df].info()

<class 'pandas.DataFrame'>
RangeIndex: 46767 entries, 0 to 46766
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Tipo identificación                 46767 non-null  str           
 1   Número de identificación            46767 non-null  int64         
 2   Saldo a fecha                       46767 non-null  float64       
 3   Valor Aporte/Contribución Mensual   46767 non-null  int64         
 4   AP                                  46767 non-null  float64       
 5   Aporte/Contribución ExtraOrdinario  46767 non-null  int64         
 6   ValorRevalorizacion                 46767 non-null  int64         
 7   Monto Promedio                      46767 non-null  int64         
 8   UltimaFecha                         46767 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(5), str(1)
memory usage: 3.2 MB


In [21]:
# Fusionar el dataframe df_features con el dataframe aportes[nombre_df] usando como clave primaria la columna 'NIT' de df_features y la columna 'Número de identificación' de aportes[nombre_df], usando el método left
df_features = pd.merge(
    df_features, 
    aportes[nombre_df][['Número de identificación', 'AP']], 
    left_on='NIT', 
    right_on='Número de identificación', 
    how='left'
    )

df_features = df_features.drop(columns=['Número de identificación'])
df_features

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP
0,002-001-0000024-8,901-285-696-7,146515.0,0.239269,1.199857e+07,0.0,1,0.0,1.0,0.45,1.0,1448.0,NaN
1,002-001-0000026-6,900-698-421-8,146515.0,0.528942,5.395877e+07,38.0,0,16.0,1.0,0.45,1.0,1181.0,NaN
2,002-001-0000027-5,811-021-864-9,146515.0,0.685802,1.632600e+09,6.0,0,6.0,1.0,0.45,1.0,1791.0,NaN
3,002-001-0000028-4,900-409-773-7,146515.0,0.508649,1.027004e+08,35.0,0,5.0,1.0,0.45,1.0,1141.0,NaN
4,002-001-0000029-3,901-406-831-6,146525.0,0.848534,6.309995e+08,91.0,0,91.0,1.0,0.60,1.0,1131.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,1035912774,144210.0,0.936706,9.806260e+05,44.0,0,44.0,1.0,0.45,1.0,1457.0,2418121.00
21016,007-002-0001464-1,15437544,144205.0,0.400000,4.066557e+06,0.0,1,0.0,1.0,0.45,1.0,6405.0,2020640.00
21017,007-002-0001467-8,43907525,144205.0,0.880690,2.684521e+06,0.0,1,0.0,1.0,0.45,1.0,463.0,775001.00
21018,007-002-0001468-7,1036964056,144205.0,0.940565,4.851986e+06,14.0,0,14.0,1.0,0.45,1.0,564.0,945001.00


# Ahorro CDAT (COOCDAT)

In [22]:
archivo = data_dir / 'INFORME INDIVIDUAL DE LAS CAPTACIONES (MODIFICADO)_SEPTIEMBRE_2025.xlsx'
nombre_df = f'captacion_{mes_anno}'
captaciones = {}
captaciones[nombre_df] = pd.read_excel(
    archivo,
    header=3
)

captaciones[nombre_df].info()

<class 'pandas.DataFrame'>
RangeIndex: 57345 entries, 0 to 57344
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   TipoIden             57345 non-null  str           
 1   NIT                  57345 non-null  object        
 2   CodigoContable       57345 non-null  int64         
 3   NombreDeposito       57345 non-null  str           
 4   TipoAhorro           57345 non-null  int64         
 5   Amortizacion         57345 non-null  int64         
 6   FechaApertura        57345 non-null  datetime64[us]
 7   Plazo                57345 non-null  int64         
 8   FechaVencimiento     32103 non-null  datetime64[us]
 9   Modalidad            57345 non-null  int64         
 10  TasaInteresNominal   57345 non-null  float64       
 11  TasaInteresEfectiva  57345 non-null  float64       
 12  InteresesCausados    57345 non-null  int64         
 13  Saldo                57345 non-null  float

In [23]:
# Crear una columna llamada 'es CDAT' que sea igual a 1 si la columna 'NombreDeposito' contiene la palabra 'CDAT' y 0 en caso contrario
captaciones[nombre_df]['es CDAT'] = captaciones[nombre_df]['NombreDeposito'].str.contains('CDAT', case=False, na=False).astype(int)
captaciones[nombre_df]['COOCDAT'] = captaciones[nombre_df]['es CDAT'] * captaciones[nombre_df]['Saldo']
captaciones[nombre_df] = captaciones[nombre_df][captaciones[nombre_df]['es CDAT']==1]
captaciones[nombre_df] = captaciones[nombre_df][['NIT', 'COOCDAT']]

# Agrupar por NIT y sumar COOCDAT
captaciones[nombre_df] = captaciones[nombre_df].groupby('NIT').sum().reset_index()
# Mostrar la información del dataframe 
captaciones[nombre_df].info()

<class 'pandas.DataFrame'>
RangeIndex: 4054 entries, 0 to 4053
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   NIT      4054 non-null   object 
 1   COOCDAT  4054 non-null   float64
dtypes: float64(1), object(1)
memory usage: 63.5+ KB


In [24]:
captaciones[nombre_df] = captaciones[nombre_df].rename(columns={'NIT': 'NIT_1'})
df_all_features = pd.merge(
    df_features, 
    captaciones[nombre_df], 
    left_on='NIT', 
    right_on='NIT_1', 
    how='left'
    )

df_all_features = df_all_features.drop(columns=['NIT_1'])
df_all_features

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP,COOCDAT
0,002-001-0000024-8,901-285-696-7,146515.0,0.239269,1.199857e+07,0.0,1,0.0,1.0,0.45,1.0,1448.0,NaN,NaN
1,002-001-0000026-6,900-698-421-8,146515.0,0.528942,5.395877e+07,38.0,0,16.0,1.0,0.45,1.0,1181.0,NaN,NaN
2,002-001-0000027-5,811-021-864-9,146515.0,0.685802,1.632600e+09,6.0,0,6.0,1.0,0.45,1.0,1791.0,NaN,NaN
3,002-001-0000028-4,900-409-773-7,146515.0,0.508649,1.027004e+08,35.0,0,5.0,1.0,0.45,1.0,1141.0,NaN,NaN
4,002-001-0000029-3,901-406-831-6,146525.0,0.848534,6.309995e+08,91.0,0,91.0,1.0,0.60,1.0,1131.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,1035912774,144210.0,0.936706,9.806260e+05,44.0,0,44.0,1.0,0.45,1.0,1457.0,2418121.00,NaN
21016,007-002-0001464-1,15437544,144205.0,0.400000,4.066557e+06,0.0,1,0.0,1.0,0.45,1.0,6405.0,2020640.00,NaN
21017,007-002-0001467-8,43907525,144205.0,0.880690,2.684521e+06,0.0,1,0.0,1.0,0.45,1.0,463.0,775001.00,NaN
21018,007-002-0001468-7,1036964056,144205.0,0.940565,4.851986e+06,14.0,0,14.0,1.0,0.45,1.0,564.0,945001.00,NaN


In [25]:
df_all_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 21020 entries, 0 to 21019
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NroCredito      21020 non-null  str    
 1   NIT             21020 non-null  object 
 2   CodigoContable  21020 non-null  float64
 3   SALPRES         21020 non-null  float64
 4   VEA             21020 non-null  float64
 5   MORA12          21020 non-null  float64
 6   SINMORA         21020 non-null  int64  
 7   Morosidad       21020 non-null  float64
 8   clasegarantia   21020 non-null  float64
 9   PDI             21020 non-null  float64
 10  Activo          21016 non-null  float64
 11  ANTI            21016 non-null  float64
 12  AP              19970 non-null  float64
 13  COOCDAT         669 non-null    float64
dtypes: float64(11), int64(1), object(1), str(1)
memory usage: 2.2+ MB


# Estimaciones de calificación crédito con libranza y sin libranza

In [26]:
cod_con_libranza = [141105, 141110, 141115, 141120, 144105, 144110, 144115, 144120, 144125]
cod_sin_libranza = [141205, 141210, 141215, 141220, 141225, 144205, 144210, 144215, 144220, 144225]

In [27]:
df_con_libranza = df_all_features[df_all_features['CodigoContable'].isin(cod_con_libranza)].copy()
df_con_libranza.info()

<class 'pandas.DataFrame'>
Index: 7787 entries, 38 to 20649
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NroCredito      7787 non-null   str    
 1   NIT             7787 non-null   object 
 2   CodigoContable  7787 non-null   float64
 3   SALPRES         7787 non-null   float64
 4   VEA             7787 non-null   float64
 5   MORA12          7787 non-null   float64
 6   SINMORA         7787 non-null   int64  
 7   Morosidad       7787 non-null   float64
 8   clasegarantia   7787 non-null   float64
 9   PDI             7787 non-null   float64
 10  Activo          7787 non-null   float64
 11  ANTI            7787 non-null   float64
 12  AP              7786 non-null   float64
 13  COOCDAT         113 non-null    float64
dtypes: float64(11), int64(1), object(1), str(1)
memory usage: 912.5+ KB


In [28]:
df_con_libranza['COOCDAT']= df_con_libranza['COOCDAT'].fillna(0)
df_con_libranza = df_con_libranza[df_con_libranza['Activo'] == 1]
df_con_libranza['AP'] = df_con_libranza['AP'].fillna(0)
df_con_libranza.info()

<class 'pandas.DataFrame'>
Index: 7786 entries, 38 to 20649
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NroCredito      7786 non-null   str    
 1   NIT             7786 non-null   object 
 2   CodigoContable  7786 non-null   float64
 3   SALPRES         7786 non-null   float64
 4   VEA             7786 non-null   float64
 5   MORA12          7786 non-null   float64
 6   SINMORA         7786 non-null   int64  
 7   Morosidad       7786 non-null   float64
 8   clasegarantia   7786 non-null   float64
 9   PDI             7786 non-null   float64
 10  Activo          7786 non-null   float64
 11  ANTI            7786 non-null   float64
 12  AP              7786 non-null   float64
 13  COOCDAT         7786 non-null   float64
dtypes: float64(11), int64(1), object(1), str(1)
memory usage: 912.4+ KB


In [29]:
def get_models_path(relative_path: str = "") -> Path:
    """Retorna la ruta absoluta a la carpeta de modelos compartida."""
    # Busca la raíz del repositorio basándose en este archivo (src/config.py)
    root = Path.cwd().parent
    models_dir = root / relative_path
    
    if not models_dir.exists():
        os.makedirs(models_dir)
        
    return models_dir

In [ ]:
# Cargar modelo 'lgbm_model_con_libranza.pkl' y crear en el dataframe 'df_con_libranza' una columna llamada Z con las predicciones del modelo
models_dir = get_models_path('models')
vars_con_libranza = ['AP', 'COOCDAT', 'MORA12', 'SINMORA', 'ANTI']
lgbm_model_con_libranza = joblib.load(models_dir / 'lightgbm_con_libranza.joblib')
df_con_libranza['Calif.modelo'] = lgbm_model_con_libranza.predict(df_con_libranza[vars_con_libranza])
df_con_libranza

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP,COOCDAT,Calif.modelo
38,002-002-0096598-0,71648448,144105.0,0.030641,899571.0,0.0,1,0.0,1.0,0.45,1.0,5445.0,5844817.0,0.0,0
44,002-002-0098148-1,98488766,144105.0,0.283385,9285591.0,0.0,1,0.0,1.0,0.45,1.0,5414.0,7105694.0,0.0,0
49,002-002-0099325-4,98595907,144105.0,0.682244,52568339.0,11.0,0,11.0,1.0,0.45,1.0,4837.0,24388732.0,0.0,0
106,002-002-0108169-2,43983251,144105.0,0.212897,2997504.0,0.0,1,0.0,1.0,0.45,1.0,6542.0,7448554.0,0.0,0
118,002-002-0108733-9,71648785,141105.0,0.053903,960344.0,8.0,0,8.0,1.0,0.45,1.0,7699.0,10945912.5,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20638,006-002-0036302-9,86061031,144105.0,1.000000,33023529.0,0.0,1,0.0,1.0,0.45,1.0,5425.0,11004175.0,0.0,0
20641,006-002-0036305-6,15431170,144105.0,1.000000,1653420.0,0.0,1,0.0,1.0,0.45,1.0,516.0,455907.0,0.0,0
20644,006-002-0036312-7,71382018,144105.0,1.000000,10043347.0,0.0,1,0.0,1.0,0.45,1.0,3202.0,3545425.0,0.0,0
20645,006-002-0036315-4,10126913,141105.0,1.000000,1425950.0,0.0,1,0.0,1.0,0.45,1.0,2201.0,1544370.0,0.0,0


In [31]:
df_con_libranza['MORA12'].describe()

count    7786.000000
mean        1.944387
std        17.189058
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       843.000000
Name: MORA12, dtype: float64

In [32]:
df_con_libranza['Calif.modelo'].value_counts()

Calif.modelo
0    7643
1      87
2      56
Name: count, dtype: int64

In [33]:
df_con_libranza['Calif.modelo'] = df_con_libranza['Calif.modelo'].map({0: 'A', 1: 'B', 2: 'C', 3:'D'})

In [34]:
# Crear columna 'Calif.homologada', que se calcula así:
# Si 'Calif.modelo' es A, entonces 'Calif.homologada' es A
# Si 'Calif.modelo' es B, y 'MORA12' es menor o igual a 30, entonces 'Calif.homologada' es A
# Si 'Calif.modelo' es B, y 'MORA12' es mayor a 30, entonces 'Calif.homologada' es B
# Si 'Calif.modelo' es C, y 'MORA12' es menor o igual a 30, entonces 'Calif.homologada' es B
# Si 'Calif.modelo' es C, y 'MORA12' es mayor a 30, entonces 'Calif.homologada' es C
# Si 'Calif.modelo' es D o E, entonces 'Calif.homologada' es C
# Si 'MORA12' es mayor o igual a 90, y menor o igual a 180, entonces 'Calif.homologada' es D
# Si 'MORA12' es mayor a 180, entonces 'Calif.homologada' es E

condiciones = [
    (df_con_libranza['Calif.modelo'] == 'A'),
    (df_con_libranza['Calif.modelo'] == 'B') & (df_con_libranza['MORA12'] <= 30),
    (df_con_libranza['Calif.modelo'] == 'B') & (df_con_libranza['MORA12'] > 30) & (df_con_libranza['MORA12'] < 90),
    (df_con_libranza['Calif.modelo'] == 'C') & (df_con_libranza['MORA12'] <= 30),
    (df_con_libranza['Calif.modelo'] == 'C') & (df_con_libranza['MORA12'] > 30) & (df_con_libranza['MORA12'] < 90),
    (df_con_libranza['Calif.modelo'].isin(['D', 'E'])),
    (df_con_libranza['MORA12'].between(90, 180))
]

opciones = ['A', 'A', 'B', 'B', 'C', 'C', 'D']

df_con_libranza['Calif.homologada'] = np.select(condiciones, opciones, default='E')

In [35]:
df_con_libranza['Calif.homologada'].value_counts()

Calif.homologada
A    7643
B      87
C      24
D      22
E      10
Name: count, dtype: int64

## Sin libranza

In [37]:
df_sin_libranza = df_all_features[df_all_features['CodigoContable'].isin(cod_sin_libranza)].copy()
df_sin_libranza.info()

<class 'pandas.DataFrame'>
Index: 13092 entries, 9 to 21019
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NroCredito      13092 non-null  str    
 1   NIT             13092 non-null  object 
 2   CodigoContable  13092 non-null  float64
 3   SALPRES         13092 non-null  float64
 4   VEA             13092 non-null  float64
 5   MORA12          13092 non-null  float64
 6   SINMORA         13092 non-null  int64  
 7   Morosidad       13092 non-null  float64
 8   clasegarantia   13092 non-null  float64
 9   PDI             13092 non-null  float64
 10  Activo          13090 non-null  float64
 11  ANTI            13090 non-null  float64
 12  AP              12102 non-null  float64
 13  COOCDAT         550 non-null    float64
dtypes: float64(11), int64(1), object(1), str(1)
memory usage: 1.5+ MB


In [38]:
df_sin_libranza[['Activo', 'ANTI', 'AP']]= df_sin_libranza[['Activo', 'ANTI', 'AP']].fillna(0)
df_sin_libranza.info()

<class 'pandas.DataFrame'>
Index: 13092 entries, 9 to 21019
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NroCredito      13092 non-null  str    
 1   NIT             13092 non-null  object 
 2   CodigoContable  13092 non-null  float64
 3   SALPRES         13092 non-null  float64
 4   VEA             13092 non-null  float64
 5   MORA12          13092 non-null  float64
 6   SINMORA         13092 non-null  int64  
 7   Morosidad       13092 non-null  float64
 8   clasegarantia   13092 non-null  float64
 9   PDI             13092 non-null  float64
 10  Activo          13092 non-null  float64
 11  ANTI            13092 non-null  float64
 12  AP              13092 non-null  float64
 13  COOCDAT         550 non-null    float64
dtypes: float64(11), int64(1), object(1), str(1)
memory usage: 1.5+ MB


In [40]:
vars_sin_libranza = ['Activo', 'ANTI', 'SALPRES', 'AP', 'MORA12']

lgbm_model_sin_libranza = joblib.load(models_dir / 'lightgbm_sin_libranza.joblib')
df_sin_libranza['Calif.modelo'] = lgbm_model_sin_libranza.predict(df_sin_libranza[vars_sin_libranza])
df_sin_libranza

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP,COOCDAT,Calif.modelo
9,002-002-0013891-9,43801451,141210.0,0.264425,5345039.0,217.0,0,53.0,3.0,0.50,0.0,0.0,0.00,NaN,4
10,002-002-0048982-5,16224395,141225.0,0.310797,20012765.0,3796.0,0,3796.0,1.0,1.00,0.0,0.0,0.00,NaN,4
11,002-002-0054500-6,70130524,144225.0,0.912728,18820457.0,3560.0,0,3560.0,1.0,1.00,0.0,0.0,0.00,NaN,4
12,002-002-0063582-8,32523503,144205.0,0.044911,922145.0,0.0,1,0.0,1.0,0.45,0.0,0.0,0.00,NaN,0
13,002-002-0067978-1,71338874,144225.0,0.717959,7107791.0,3963.0,0,3963.0,1.0,1.00,1.0,6668.0,3146429.00,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,1035912774,144210.0,0.936706,980626.0,44.0,0,44.0,1.0,0.45,1.0,1457.0,2418121.00,NaN,1
21016,007-002-0001464-1,15437544,144205.0,0.400000,4066557.0,0.0,1,0.0,1.0,0.45,1.0,6405.0,2020640.00,NaN,0
21017,007-002-0001467-8,43907525,144205.0,0.880690,2684521.0,0.0,1,0.0,1.0,0.45,1.0,463.0,775001.00,NaN,0
21018,007-002-0001468-7,1036964056,144205.0,0.940565,4851986.0,14.0,0,14.0,1.0,0.45,1.0,564.0,945001.00,NaN,0


In [41]:
df_sin_libranza['Calif.modelo'] = df_sin_libranza['Calif.modelo'].map({0: 'A', 1: 'B', 2: 'C', 3:'D', 4:'E'})

In [42]:
condiciones = [
    (df_sin_libranza['Calif.modelo'] == 'A'),
    (df_sin_libranza['Calif.modelo'] == 'B') & (df_sin_libranza['MORA12'] <= 30),
    (df_sin_libranza['Calif.modelo'] == 'B') & (df_sin_libranza['MORA12'] > 30) & (df_sin_libranza['MORA12'] < 90),
    (df_sin_libranza['Calif.modelo'] == 'C') & (df_sin_libranza['MORA12'] <= 30),
    (df_sin_libranza['Calif.modelo'] == 'C') & (df_sin_libranza['MORA12'] > 30) & (df_sin_libranza['MORA12'] < 90),
    (df_sin_libranza['Calif.modelo'].isin(['D', 'E'])),
    (df_sin_libranza['MORA12'].between(90, 180))
]

opciones = ['A', 'A', 'B', 'B', 'C', 'C', 'D']

df_sin_libranza['Calif.homologada'] = np.select(condiciones, opciones, default='E')

In [43]:
df_sin_libranza['Calif.modelo'].value_counts()

Calif.modelo
A    7642
C    3867
E     905
B     657
D      21
Name: count, dtype: int64

In [44]:
df_sin_libranza['Calif.homologada'].value_counts()

Calif.homologada
A    7642
E    3076
C    1243
B     657
D     474
Name: count, dtype: int64

## Calificación de arrastre, calificación final y cálculo de PI y PE

In [45]:
# Crear un dataframe que tenga una columna que seea el NIT de los dataframes df_con_libranza y df_sin_libranza y 
# otra columna que sea la peor calificacion homologada de cada NIT

df_peor_calificacion = pd.concat([df_con_libranza[['NIT', 'Calif.homologada']], df_sin_libranza[['NIT', 'Calif.homologada']]])
df_peor_calificacion = df_peor_calificacion.groupby('NIT')['Calif.homologada'].max().reset_index()
df_peor_calificacion


,NIT,Calif.homologada
0,204414,A
1,219552,A
2,366107,E
3,371497,A
4,398038,A
...,...,...
17966,9004400754,A
17967,10376145374,A
17968,811-012-598-6,A
17969,890-985-579-8,A


In [46]:
# En df_con_libranza crear una columna llamada 'Calif.final' cuyos valores sean iguales a la columna 'Calif.homologada' del df_peor_calificacion de acuerdo al valor de NIT.
df_con_libranza['Calif.final'] = df_con_libranza['NIT'].map(df_peor_calificacion.set_index('NIT')['Calif.homologada'])
df_con_libranza['Calif.final'].value_counts()

Calif.final
A    7532
B     134
C      50
E      37
D      33
Name: count, dtype: int64

In [47]:
# En df_sin_libranza crear una columna llamada 'Calif.final' cuyos valores sean iguales a la columna 'Calif.homologada' del df_peor_calificacion de acuerdo al valor de NIT.
df_sin_libranza['Calif.final'] = df_sin_libranza['NIT'].map(df_peor_calificacion.set_index('NIT')['Calif.homologada'])
df_sin_libranza['Calif.final'].value_counts()

Calif.final
A    7514
E    3105
C    1277
B     710
D     486
Name: count, dtype: int64

In [48]:
# Crear una columna llamada 'PI' aue se calcula así: si calificación es A, PI = 0.005; 
# si calificación es B, PI = 0.006; si calificación es C, PI = 0.0441; 
# si calificación es D, PI = 0.0448; si calificación es E, PI = 0.2273
df_con_libranza['PI'] = df_con_libranza['Calif.final'].map({
    'A': 0.005,
    'B': 0.006,
    'C': 0.0441,
    'D': 0.0448,
    'E': 0.2273
})

df_con_libranza['PI'] = df_con_libranza['PI'].astype(float)

df_con_libranza['PE'] = df_con_libranza['PI'] * df_con_libranza['VEA'] * df_con_libranza['PDI']

df_con_libranza

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP,COOCDAT,Calif.modelo,Calif.homologada,Calif.final,PI,PE
38,002-002-0096598-0,71648448,144105.0,0.030641,899571.0,0.0,1,0.0,1.0,0.45,1.0,5445.0,5844817.0,0.0,A,A,A,0.005,2024.03475
44,002-002-0098148-1,98488766,144105.0,0.283385,9285591.0,0.0,1,0.0,1.0,0.45,1.0,5414.0,7105694.0,0.0,A,A,A,0.005,20892.57975
49,002-002-0099325-4,98595907,144105.0,0.682244,52568339.0,11.0,0,11.0,1.0,0.45,1.0,4837.0,24388732.0,0.0,A,A,A,0.005,118278.76275
106,002-002-0108169-2,43983251,144105.0,0.212897,2997504.0,0.0,1,0.0,1.0,0.45,1.0,6542.0,7448554.0,0.0,A,A,A,0.005,6744.38400
118,002-002-0108733-9,71648785,141105.0,0.053903,960344.0,8.0,0,8.0,1.0,0.45,1.0,7699.0,10945912.5,0.0,A,A,A,0.005,2160.77400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20638,006-002-0036302-9,86061031,144105.0,1.000000,33023529.0,0.0,1,0.0,1.0,0.45,1.0,5425.0,11004175.0,0.0,A,A,A,0.005,74302.94025
20641,006-002-0036305-6,15431170,144105.0,1.000000,1653420.0,0.0,1,0.0,1.0,0.45,1.0,516.0,455907.0,0.0,A,A,A,0.005,3720.19500
20644,006-002-0036312-7,71382018,144105.0,1.000000,10043347.0,0.0,1,0.0,1.0,0.45,1.0,3202.0,3545425.0,0.0,A,A,A,0.005,22597.53075
20645,006-002-0036315-4,10126913,141105.0,1.000000,1425950.0,0.0,1,0.0,1.0,0.45,1.0,2201.0,1544370.0,0.0,A,A,A,0.005,3208.38750


In [49]:
# Crear una columna llamada 'PI' aue se calcula así: si calificación es A, PI = 0.005; 
# si calificación es B, PI = 0.006; si calificación es C, PI = 0.0441; 
# si calificación es D, PI = 0.0448; si calificación es E, PI = 0.2273
df_sin_libranza['PI'] = df_sin_libranza['Calif.final'].map({
    'A': 0.015,
    'B': 0.0595,
    'C': 0.1382,
    'D': 0.3277,
    'E': 0.4171
})

df_sin_libranza['PI'] = df_sin_libranza['PI'].astype(float)

df_sin_libranza['PE'] = df_sin_libranza['PI'] * df_sin_libranza['VEA'] * df_sin_libranza['PDI']

df_sin_libranza

,NroCredito,NIT,CodigoContable,SALPRES,VEA,MORA12,SINMORA,Morosidad,clasegarantia,PDI,Activo,ANTI,AP,COOCDAT,Calif.modelo,Calif.homologada,Calif.final,PI,PE
9,002-002-0013891-9,43801451,141210.0,0.264425,5345039.0,217.0,0,53.0,3.0,0.50,0.0,0.0,0.00,NaN,E,C,C,0.1382,3.693422e+05
10,002-002-0048982-5,16224395,141225.0,0.310797,20012765.0,3796.0,0,3796.0,1.0,1.00,0.0,0.0,0.00,NaN,E,C,C,0.1382,2.765764e+06
11,002-002-0054500-6,70130524,144225.0,0.912728,18820457.0,3560.0,0,3560.0,1.0,1.00,0.0,0.0,0.00,NaN,E,C,C,0.1382,2.600987e+06
12,002-002-0063582-8,32523503,144205.0,0.044911,922145.0,0.0,1,0.0,1.0,0.45,0.0,0.0,0.00,NaN,A,A,A,0.0150,6.224479e+03
13,002-002-0067978-1,71338874,144225.0,0.717959,7107791.0,3963.0,0,3963.0,1.0,1.00,1.0,6668.0,3146429.00,NaN,C,E,E,0.4171,2.964660e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21015,007-002-0001463-2,1035912774,144210.0,0.936706,980626.0,44.0,0,44.0,1.0,0.45,1.0,1457.0,2418121.00,NaN,B,B,B,0.0595,2.625626e+04
21016,007-002-0001464-1,15437544,144205.0,0.400000,4066557.0,0.0,1,0.0,1.0,0.45,1.0,6405.0,2020640.00,NaN,A,A,A,0.0150,2.744926e+04
21017,007-002-0001467-8,43907525,144205.0,0.880690,2684521.0,0.0,1,0.0,1.0,0.45,1.0,463.0,775001.00,NaN,A,A,A,0.0150,1.812052e+04
21018,007-002-0001468-7,1036964056,144205.0,0.940565,4851986.0,14.0,0,14.0,1.0,0.45,1.0,564.0,945001.00,NaN,A,A,A,0.0150,3.275091e+04


In [50]:
# Guardar en archivo csv llamado 'df_con_libranza_calificaciones_septiembre_2025.csv' el dataframe df_con_libranza
df_con_libranza.to_csv(data_dir / 'df_con_libranza_calificaciones_septiembre_2025.csv', index=False)

In [51]:
# Guardar en archivo csv llamado 'df_sin_libranza_calificaciones_septiembre_2025.csv' el dataframe df_sin_libranza
df_sin_libranza.to_csv(data_dir / 'df_sin_libranza_calificaciones_septiembre_2025.csv', index=False)